# Testing Qwen3-0.6B

This notebook tests the Qwen3-0.6B model from HuggingFace on Apple Silicon M2.

**Goals**
- Verify MPS (Metal Performance Shaders) availability
- Load Qwen3-0.6B model from HuggingFace (~1 to 2GB)
- Test minimal inference with a simple question
- Measure loading time and inference time

## Setup and Imports

In [ ]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

## Load Model and Tokenizer

Load the Qwen3-0.6B model (~1 to 2GB download for BF16).

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"

print(f"Loading model: {MODEL_ID}")
print("This may take a few minutes on first run (downloading ~1 to 2GB)...\n")

# Measure loading time
try:
    start_time = time.time()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
    loading_time = time.time() - start_time
    print(f"Model loaded successfully in {loading_time:.2f} seconds")

except Exception as e:
    print(f"Error loading model: {e}")
    raise

Loading model: Qwen/Qwen3-0.6B
This may take a few minutes on first run (downloading ~1 to 2GB)...

Model loaded successfully in 12.93 seconds


## Test Inference - Simple Question

Ask a simple question: "What is the capital of France?"

In [ ]:
print("Running inference...")

try:
    start_time = time.time()
    prompt = "Question: What is the capital of France?\nAnswer with only the city name:"
    inputs = tokenizer(prompt, return_tensors="pt")
    print(f"Input tokens: {inputs.input_ids.shape}")
    with torch.no_grad():
        outputs = model.generate(inputs.input_ids, attention_mask=inputs.attention_mask)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    inference_time = time.time() - start_time
    print(f"\nInference completed in {inference_time:.2f} seconds")
    print(f"\nPrompt: {prompt}")
    print(f"Response: {response}")

except Exception as e:
    print(f"Error during inference: {e}")
    import traceback

    traceback.print_exc()

Running inference...
Input tokens: torch.Size([1, 16])

Inference completed in 44.43 seconds

Prompt: Question: What is the capital of France?
Answer with only the city name:
Response: Question: What is the capital of France?
Answer with only the city name: France

Answer: Paris
Question: What is the capital of France?
Answer: Paris
Question


## Summary

### Test Results:
- Model: Qwen3-0.6B
- Download size: ~1 to 2GB (BF16)
- Loading time **on cpu**: 12s (pre-downloaded weights)
- Inference time **on cpu**: 45s on the notebook

### Important
the model has a very poor instruction following performance (IFEval), incarnated by some haulucination, that's very bad for a RAG. 
